# Correction des prévisions de vent par gradient boosting (Chémoulin)

**Question :** un modèle de ML peut-il corriger le vent prévu par AROME et GFS mieux qu'une simple correction de biais ?

**Démarche**
- **Trois prévisions à corriger :** AROME, GFS et la moyenne des deux. Pour chacune, le modèle apprend l'**erreur** (prévision − mesure, même convention que `src/metrics.py`).
  Le vent corrigé (« reconstruit ») vaut ensuite prévision − erreur prédite. On compare les variantes sur ce **vent reconstruit**, pas sur la qualité d'ajustement de l'erreur.
- **Variables d'entrée (features) :** uniquement des valeurs **prévues**, jamais mesurées (sinon le modèle « triche »).
- **Découpage temporel :** train de nov. 2022 à oct. 2025, test de nov. 2025 à oct. 2026.
  On fait une validation croisée par année entière sur le train ; le test reste **fermé** jusqu'à la toute fin.
- **Bases de comparaison (baselines) :** prévision brute, puis prévision corrigée de son biais moyen.
- **Vérité terrain :** station Météo-France de la Pointe de Chémoulin.

Le travail avance étape par étape : on lit les résultats de chaque cellule avant de passer à la suivante.

## 1. Chargement des données

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

# Le notebook peut être lancé depuis notebooks/ ou depuis la racine du projet
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from metrics import is_navigable  # même définition de « navigable » que dans l'analyse d'erreurs

RESULTS_DIR = ROOT / "results"

In [2]:
df = pd.read_csv(ROOT / "data/processed/dataset_chemoulin.csv", parse_dates=["time"])
df = df.set_index("time").sort_index()
df = df.asfreq("h")  # grille horaire continue : nécessaire pour les fenêtres glissantes (min/max sur ±3 h)

# Codes qualité Météo-France : 0 et 1 = validé, 9 = pas encore validé (gardé, sinon on perd
# la fin du test). Les autres codes (douteux) sont écartés.
GOOD_Q = [0, 1, 9]
df.loc[~df["q_speed"].isin(GOOD_Q), "speed_obs"] = np.nan
df.loc[~df["q_direction"].isin(GOOD_Q), "dir_obs"] = np.nan

print(df.shape)
df["q_speed"].value_counts(dropna=False)

(51149, 19)


q_speed
1.0    50391
NaN      738
9.0       20
Name: count, dtype: int64

## 2. Features : uniquement des valeurs prévues

| Feature | Ce qu'elle apporte |
|---|---|
| `speed_*` | vent moyen prévu par AROME et par GFS |
| `gust_*` | rafales prévues : régularité du vent à l'instant t |
| `min_pm3h_*`, `max_pm3h_*` | min et max du vent prévu de t−3 h à t+3 h : variabilité (bascule, renforcement) |
| `dir_sin_*`, `dir_cos_*` | direction codée sur le cercle : 359° et 1° deviennent voisins |
| `heure_sin/cos`, `mois_sin/cos` | cycle journalier et saisonnier : 23 h et 0 h sont voisins, tout comme décembre et janvier |
| `ecart_arome_gfs` | désaccord entre les modèles : s'ils sont d'accord, le phénomène est de grande échelle ; sinon, il est local (thermique, effet de côte) |

La fenêtre ±3 h ne crée pas de fuite : ce sont des valeurs **prévues**, toutes connues au moment où l'on consulte la prévision.

Pourquoi sin/cos ? Codé brut, 359° est « loin » de 1°. Avec (sin, cos), chaque angle devient un point du cercle, et deux angles proches donnent deux points proches.

In [ ]:
MODELS = ["arome", "gfs"]
WINDOW = 7  # 7 heures centrées : de t-3 h à t+3 h


def cyclic(values, period):
    """Code une valeur cyclique (heure, mois, angle) en deux coordonnées sur un cercle."""
    angle = 2 * np.pi * values / period
    return np.sin(angle), np.cos(angle)


def build_features(df):
    X = pd.DataFrame(index=df.index)
    for m in MODELS:
        speed = df[f"speed_{m}"]
        X[f"speed_{m}"] = speed
        X[f"gust_{m}"] = df[f"gust_{m}"]
        window = speed.rolling(WINDOW, center=True, min_periods=4)
        X[f"min_pm3h_{m}"] = window.min()
        X[f"max_pm3h_{m}"] = window.max()
        X[f"dir_sin_{m}"], X[f"dir_cos_{m}"] = cyclic(df[f"dir_{m}"], 360)
    X["ecart_arome_gfs"] = df["speed_arome"] - df["speed_gfs"]
    X["heure_sin"], X["heure_cos"] = cyclic(df.index.hour, 24)
    X["mois_sin"], X["mois_cos"] = cyclic(df.index.month - 1, 12)
    return X


# Calculées sur la série complète AVANT de filtrer les lignes, pour que les fenêtres ±3 h
# voient bien les heures voisines
X_all = build_features(df)
FEATURES = list(X_all.columns)
FEATURES

## 3. Périmètre, cibles, découpage train / test

- On garde les heures où la mesure ET les deux prévisions (vitesse, direction) existent. Les rafales peuvent manquer : le gradient boosting de scikit-learn gère les valeurs manquantes.
- **Années de validation** : de novembre à octobre, l'année « 2023 » allant de nov. 2022 à oct. 2023. AROME commence le 13/11/2022, donc la première année est un peu plus courte.
- **Heures navigables** : vent ≥ 12 nœuds et secteur 135° à 292,5°, **mesuré OU prévu** par AROME ou GFS. C'est le même principe que dans `metrics.py`. Ce masque est fixé une fois pour toutes, donc toutes les méthodes sont jugées exactement sur les mêmes heures.

In [3]:
START = pd.Timestamp("2022-11-01", tz="UTC")
SPLIT = pd.Timestamp("2025-11-01", tz="UTC")
END = pd.Timestamp("2026-11-01", tz="UTC")

df["speed_moy"] = (df["speed_arome"] + df["speed_gfs"]) / 2
VARIANTS = {"arome": "speed_arome", "gfs": "speed_gfs", "moy": "speed_moy"}

needed = ["speed_obs", "speed_arome", "speed_gfs", "dir_arome", "dir_gfs"]
data = df[(df.index >= START) & (df.index < END)].dropna(subset=needed).copy()

data["annee"] = data.index.year + (data.index.month >= 11)  # nov. 2022 -> oct. 2023 = 2023
data["nav"] = (is_navigable(data["speed_obs"], data["dir_obs"])
               | is_navigable(data["speed_arome"], data["dir_arome"])
               | is_navigable(data["speed_gfs"], data["dir_gfs"]))

train = data[data.index < SPLIT]
test = data[data.index >= SPLIT]  # on n'y touche plus jusqu'à la section 6

print("train :", train.shape, "| test :", test.shape)
train.groupby("annee").agg(heures=("speed_obs", "size"), part_navigable=("nav", "mean")).round(3)

train : (25873, 22) | test : (8758, 22)


,heures,part_navigable
annee,,
2023,8459,0.344
2024,8664,0.376
2025,8750,0.263


In [ ]:
# Contrôle des features sur le train : valeurs manquantes et ordres de grandeur
X_all.loc[train.index].isna().mean().round(3)

In [ ]:
X_all.loc[train.index].describe().T.round(2)

## 4. Outils d'évaluation et validation croisée

**Validation croisée « leave-one-year-out »** : 3 plis. On entraîne sur 2 années et on valide sur la 3e, à tour de rôle.
On découpe par années entières, jamais au hasard. Sinon, 14 h et 15 h du même jour (quasi identiques) pourraient tomber l'une dans le train, l'autre dans la validation : le modèle aurait l'air bon simplement parce qu'il a déjà vu cette situation (fuite de données).

Chaque méthode reçoit (train, validation, colonne de prévision) et renvoie le **vent reconstruit** sur la validation.

In [ ]:
def scores(d, pred):
    """Biais, MAE, RMSE du vent reconstruit, sur toutes les heures et sur les heures navigables."""
    obs = d["speed_obs"].to_numpy()
    rows = []
    for subset, mask in [("toutes", np.ones(len(d), dtype=bool)),
                         ("navigables", d["nav"].to_numpy())]:
        err = pred[mask] - obs[mask]
        rows.append({"sous-ensemble": subset, "heures": int(mask.sum()),
                     "biais (kn)": err.mean(),
                     "MAE (kn)": np.abs(err).mean(),
                     "RMSE (kn)": np.sqrt((err ** 2).mean())})
    return rows


def cross_validate(methods):
    """Évalue chaque méthode sur chaque prévision (arome, gfs, moy) et chaque année de validation."""
    rows = []
    for year in sorted(train["annee"].unique()):
        tr, va = train[train["annee"] != year], train[train["annee"] == year]
        for variant, col in VARIANTS.items():
            for method_name, method in methods.items():
                pred = method(tr, va, col)
                for r in scores(va, pred):
                    rows.append({"année validée": year, "prévision": variant,
                                 "méthode": method_name, **r})
    return pd.DataFrame(rows)


def summarize(cv):
    """Moyenne des 3 plis, par sous-ensemble, prévision et méthode."""
    return (cv.groupby(["sous-ensemble", "prévision", "méthode"])
              [["biais (kn)", "MAE (kn)", "RMSE (kn)"]].mean().round(2))

## 5a. Baselines : prévision brute et correction du biais moyen

- **Brut** : erreur prédite = 0, on garde la prévision telle quelle.
- **Biais moyen** : on retire l'erreur moyenne observée sur les années d'entraînement. C'est la correction la plus simple possible, celle que le gradient boosting doit battre pour justifier sa complexité.

In [ ]:
def brut(tr, va, col):
    return va[col].to_numpy()


def biais_moyen(tr, va, col):
    bias = (tr[col] - tr["speed_obs"]).mean()
    return va[col].to_numpy() - bias


cv_base = cross_validate({"brut": brut, "biais moyen": biais_moyen})
summarize(cv_base)

**Lecture des résultats.** Les scores bruts sont les mêmes que dans l'analyse d'erreurs, à moins de 0,1 kn près. Les petits écarts viennent de la période et des heures communes à AROME et GFS.

La correction par le biais moyen n'améliore presque rien : sur les heures navigables, la MAE baisse de 0,5 kn pour AROME mais augmente de 0,3 kn pour GFS. En effet, les modèles **surestiment les vents faibles et sous-estiment les vents forts**. Sur l'ensemble des heures, ces deux erreurs se compensent, donc le biais global est faible (−0,8 kn pour AROME, +0,5 kn pour GFS). Le retirer revient à décaler toutes les prévisions d'une petite valeur constante, alors que l'erreur dépend de la force du vent. Pour GFS, cette correction baisse même la prévision alors que le modèle sous-estime déjà les vents navigables.

Ce baseline est donc trop facile à battre. On ajoute un baseline plus pertinent : une **régression linéaire** du vent mesuré sur le vent prévu. Elle corrige davantage les vents forts que les vents faibles. Si le gradient boosting ne fait pas mieux qu'une simple droite, sa complexité n'est pas justifiée.

In [ ]:
def regression_lineaire(tr, va, col):
    # Droite vent mesuré = a × vent prévu + b, ajustée sur les années d'entraînement
    a, b = np.polyfit(tr[col], tr["speed_obs"], deg=1)
    return a * va[col].to_numpy() + b


cv_base = cross_validate({"brut": brut, "biais moyen": biais_moyen,
                          "régression linéaire": regression_lineaire})
summarize(cv_base)

## 5b. Gradient boosting

`HistGradientBoostingRegressor` de scikit-learn : rapide, il gère les valeurs manquantes (rafales absentes) et ne demande aucune normalisation.
Les réglages sont volontairement simples et identiques pour les trois prévisions. On les ajustera seulement si les résultats le justifient, et toujours en validation croisée.
`early_stopping=False`, car l'arrêt anticipé prendrait 10 % du train **au hasard** pour se valider, ce qu'on veut justement éviter.

In [ ]:
GB_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                 min_samples_leaf=50, early_stopping=False, random_state=0)


def gradient_boosting(tr, va, col):
    model = HistGradientBoostingRegressor(**GB_PARAMS)
    model.fit(X_all.loc[tr.index, FEATURES], tr[col] - tr["speed_obs"])  # cible : l'erreur
    predicted_error = model.predict(X_all.loc[va.index, FEATURES])
    return va[col].to_numpy() - predicted_error  # vent reconstruit


cv_gb = cross_validate({"gradient boosting": gradient_boosting})
cv = pd.concat([cv_base, cv_gb], ignore_index=True)
summary = summarize(cv)
summary

In [ ]:
# Stabilité d'une année à l'autre : MAE de chaque pli (un gain réel doit apparaître sur les 3 années)
cv.pivot_table(index=["sous-ensemble", "prévision", "méthode"],
               columns="année validée", values="MAE (kn)").round(2)

In [ ]:
summary.to_csv(RESULTS_DIR / "cv_correction_vent_chemoulin.csv")
cv.round(3).to_csv(RESULTS_DIR / "cv_correction_vent_chemoulin_par_annee.csv", index=False)

## 6. Test final (une seule fois, à la toute fin)

On réentraîne sur tout le train (nov. 2022 à oct. 2025) et on évalue sur nov. 2025 à oct. 2026.
Ne passer `EVALUER_TEST` à `True` qu'une fois les choix figés : si l'on regarde le test puis qu'on ajuste le modèle, le test ne mesure plus rien d'honnête.

In [ ]:
EVALUER_TEST = False

if EVALUER_TEST:
    methods = {"brut": brut, "biais moyen": biais_moyen,
               "régression linéaire": regression_lineaire,
               "gradient boosting": gradient_boosting}
    rows = []
    for variant, col in VARIANTS.items():
        for method_name, method in methods.items():
            for r in scores(test, method(train, test, col)):
                rows.append({"prévision": variant, "méthode": method_name, **r})
    test_scores = pd.DataFrame(rows).round(2)
    test_scores.to_csv(RESULTS_DIR / "test_correction_vent_chemoulin.csv", index=False)
    display(test_scores)